# Classificador de imagens CIFAR-10

Mesmo fluxo do notebook `deep_learning/notebooks/Classificador_Imagens.ipynb`, agora usando o pacote `module_cifar10`:
carregar a base, ver exemplos, treinar a CNN, avaliar no teste e olhar as predições e as curvas de treino.

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from module_cifar10.config import CLASS_NAMES, MODEL_PATH
from module_cifar10.dataset import load_data
from module_cifar10.modeling.evaluate import accuracy_per_class, evaluate_model
from module_cifar10.modeling.model import build_model
from module_cifar10.modeling.predict import predict_classes
from module_cifar10.modeling.train import save_model, train_model
from module_cifar10.plots import plot_history, plot_predictions, plot_samples

## Carregando a base

Na primeira execução a base é baixada e salva em `data/raw/cifar10.npz`; as imagens já vêm normalizadas em [0, 1].

In [ ]:
(X_train, y_train), (X_test, y_test) = load_data()

print(f"Formato de X_train: {X_train.shape}")
print(f"Formato de y_train: {y_train.shape}")
print(f"Formato de X_test: {X_test.shape}")
print(f"Formato de y_test: {y_test.shape}")

In [ ]:
plot_samples(X_train, y_train)
plt.show()

## Arquitetura

Data augmentation (flip e translação), três blocos conv-conv-pool com BatchNorm e Dropout (32 → 64 → 128 filtros) e um classificador denso.

In [ ]:
build_model().summary()

## Treino

Até 60 épocas com EarlyStopping (paciência 10 na `val_accuracy`) e ReduceLROnPlateau. Em CPU demora bastante; no Colab com GPU T4 leva uns 15 minutos.

In [ ]:
model, history = train_model(X_train, y_train)
save_model(model, history)

## Avaliação no teste

In [ ]:
metrics = evaluate_model(model, X_test, y_test)

print(f"Acurácia no conjunto de teste: {metrics['accuracy'] * 100:.2f}%")
print(f"Loss no teste: {metrics['loss']:.4f}")

In [ ]:
y_pred = predict_classes(model, X_test)

for name, acc in sorted(accuracy_per_class(y_test, y_pred).items(), key=lambda item: item[1]):
    print(f"{name:<10} {acc * 100:.2f}%")

In [ ]:
plot_predictions(X_test, y_test, y_pred)
plt.show()

## Curvas de treino

In [ ]:
plot_history(history)
plt.show()